In [1]:
import spacy
from datasets import load_dataset
from tqdm import tqdm
import json
from pprint import pp

/home/users/v/vollert/.conda/envs/tmqa/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys
sys.path.append('../')
import os
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# Building oracle pool

In [13]:
path_name = "qiaojin/PubMedQA"
subset = "pqa_labeled"
split = "train"

ds = load_dataset(path=path_name, name=subset, split=split, token=HF_TOKEN)

In [2]:
import torch
torch.cuda.is_available()

True

In [20]:
[item['question'] for item in ds]

['Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?',
 'Landolt C and snellen e acuity: differences in strabismus amblyopia?',
 'Syncope during bathing in infants, a pediatric form of water-induced urticaria?',
 'Are the long-term results of the transanal pull-through equal to those of the transabdominal pull-through?',
 'Can tailored interventions increase mammography use among HMO women?',
 'Double balloon enteroscopy: is it efficacious and safe in a community setting?',
 '30-Day and 1-year mortality in emergency general surgery laparotomies: an area of concern and need for improvement?',
 'Is adjustment for reporting heterogeneity necessary in sleep disorders?',
 'Do mutations causing low HDL-C promote increased carotid intima-media thickness?',
 "A short stay or 23-hour ward in a general and academic children's hospital: are they effective?",
 "Did Chile's traffic law reform push police enforcement?",
 'Therapeutic anticoagulation in the tra

In [4]:
nlp = spacy.load("en_core_sci_md")
pubmedqa_pool = []

/home/users/v/vollert/.conda/envs/tmqa/lib/python3.11/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [9]:
for entry in tqdm(ds, desc="Building PubMedQA oracle pool"):
    pubid = str(entry['pubid'])
    question = entry['question']
    long_answer = entry['long_answer']
    final_decision = entry['final_decision']

    # context.contexts is a list of paragraph-level strings
    # flatten to sentence level via scispaCy
    sentences = []
    for paragraph in entry['context']['contexts']:
        doc = nlp(paragraph)
        sentences.extend([sent.text.strip() for sent in doc.sents if sent.text.strip()])

    candidates = [
        {"sentence": s, "relevance_score": 0.0}
        for s in sentences
    ]

    pubmedqa_pool.append({
        "id": pubid,
        "query": question,
        "answer": long_answer,
        "final_decision": final_decision,
        "pubmed_ids": [pubid],
        "candidates": candidates
    })

print(f"Built pool for {len(pubmedqa_pool)} queries.")
print(f"Example entry:")
pp(pubmedqa_pool[0])

Building PubMedQA oracle pool:   0%| | 0/1000 [00:00<?, ?it/

Building PubMedQA oracle pool: 100%|█| 1000/1000 [00:40<00:0

Built pool for 2000 queries.
Example entry:
{'id': '21645374',
 'query': 'Do mitochondria play a role in remodelling lace plant leaves during '
          'programmed cell death?',
 'answer': 'Results depicted mitochondrial dynamics in vivo as PCD progresses '
           'within the lace plant, and highlight the correlation of this '
           'organelle with other organelles during developmental PCD. To the '
           'best of our knowledge, this is the first report of mitochondria '
           'and chloroplasts moving on transvacuolar strands to form a ring '
           'structure surrounding the nucleus during developmental PCD. Also, '
           'for the first time, we have shown the feasibility for the use of '
           'CsA in a whole plant system. Overall, our findings implicate the '
           'mitochondria as playing a critical and early role in '
           'developmentally regulated PCD in the lace plant.',
 'final_decision': 'yes',
 'pubmed_ids': ['21645374'],
 'candi

In [10]:
# Dump for reuse
pubmedqa_oracle_pool_path = "evaluation/pubmedqa_oracle_pool.json"
with open(pubmedqa_oracle_pool_path, "w", encoding="utf-8") as f:
    json.dump(pubmedqa_pool, f, ensure_ascii=False, indent=2)

print("Pool saved to pubmedqa_oracle_pool.json")

Pool saved to pubmedqa_oracle_pool.json


# Realistic setting

In [1]:
import requests
import json
from tqdm import tqdm
from datasets import load_dataset
import os

SIBILS_URL = "https://biodiversitypmc.sibils.org/api/search"
PUBMED_FETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

/home/users/v/vollert/.conda/envs/tmqa/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
path_name = "qiaojin/PubMedQA"
subset = "pqa_artificial"
split = "train"

ds_artificial = load_dataset(path=path_name, name=subset, split=split, token=HF_TOKEN)
ds_artificial = ds_artificial.select(range(10000))

In [6]:
def fetchAbstractsForQuery(query: str) -> list[dict]:
    params = {
        "q": query,
        "col": "medline"
    }
    try:
        r = requests.post(url=SIBILS_URL, params=params, timeout=30)
        r.raise_for_status()
        response_json = json.loads(r.text)
        hits = response_json['elastic_output']['hits']['hits']
        abstracts = []
        for ab in hits:
            source = ab["_source"]
            abstract_text = source.get("abstract", "").strip()
            if not abstract_text:
                continue
            abstracts.append({
                "id": ab["_id"],
                "sibilScore": ab["_score"],
                "title": source.get("title", ""),
                "fullTextAbstract": abstract_text
            })
        return abstracts
    except requests.exceptions.Timeout:
        print(f"Timeout for query: {query[:60]}...")
        return []
    except Exception as e:
        print(f"Error fetching query '{query[:60]}...': {e}")
        return []

    
fetchAbstractsForQuery(ds_artificial[2]["question"])

[{'id': '25445714',
  'sibilScore': 264.21637,
  'title': 'Psammaplin A induces Sirtuin 1-dependent autophagic cell death in doxorubicin-resistant MCF-7/adr human breast cancer cells and xenografts.',
  'fullTextAbstract': 'BACKGROUND\nPsammaplin A (PsA) is a natural product isolated from marine sponges, which has been demonstrated to have anticancer activity against several human cancer cell lines via the induction of cell cycle arrest and apoptosis. New drugs that are less toxic and more effective against multidrug-resistant cancers are urgently needed.\n\n\nMETHODS\nWe tested cell proliferation, cell cycle progression and autophagic cell death pathway in doxorubicin-resistant MCF-7 (MCF-7/adr) human breast cancer cells. The potency of PsA was further determined using an in vivo xenograft model.\n\n\nRESULTS AND CONCLUSION\nPsA significantly inhibited MCF-7/adr cells proliferation in a concentration-dependent manner, with accumulation of cells in G2/M phase of the cell cycle. PsA sig

In [7]:
def tokenize_abstract(text: str) -> list[str]:
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]


In [8]:
import time
nlp = spacy.load("en_core_sci_md")

# --- Main fetching loop ---
# Maps pubmed_id -> list of sentences, deduplicated across queries
abstract_list_tokenized = {}

# Maps query pubid -> list of retrieved pubmed IDs (including empty lists)
query_to_retrieved_ids = {}

for entry in tqdm(ds_artificial, desc="Fetching abstracts"):
    pubid = str(entry['pubid'])
    question = entry['question']

    fetched = fetchAbstractsForQuery(question)
    time.sleep(0.1)

    retrieved_ids = []
    for ab in fetched:
        ab_id = str(ab['id'])
        retrieved_ids.append(ab_id)

        # Only tokenize and store if not already seen
        if ab_id not in abstract_list_tokenized:
            sentences = tokenize_abstract(ab['fullTextAbstract'])
            if sentences:
                abstract_list_tokenized[ab_id] = sentences

    query_to_retrieved_ids[pubid] = retrieved_ids

# --- Save both outputs ---
with open("pubmedqa_abstract_list_tokenized.json", "w", encoding="utf-8") as f:
    json.dump(abstract_list_tokenized, f, ensure_ascii=False, indent=2)

with open("pubmedqa_query_to_retrieved_ids.json", "w", encoding="utf-8") as f:
    json.dump(query_to_retrieved_ids, f, ensure_ascii=False, indent=2)

# --- Summary ---
n_queries_with_results = sum(1 for ids in query_to_retrieved_ids.values() if ids)
n_queries_empty = sum(1 for ids in query_to_retrieved_ids.values() if not ids)
n_unique_abstracts = len(abstract_list_tokenized)

print(f"Queries processed       : {len(query_to_retrieved_ids)}")
print(f"Queries with results    : {n_queries_with_results}")
print(f"Queries with no results : {n_queries_empty}")
print(f"Unique abstracts stored : {n_unique_abstracts}")

/home/users/v/vollert/.conda/envs/tmqa/lib/python3.11/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]
Fetching abstracts:   0%|                                                           | 13/10000 [00:11<2:15:28,  1.23it/s]

Error fetching query 'Are [ Higher nitric oxide levels associated with disease act...': 'elastic_output'


Fetching abstracts:   0%|▎                                                          | 43/10000 [00:39<2:30:58,  1.10it/s]


KeyboardInterrupt: 

In [ ]:
pubmedqa_rag_pool = []
skipped_no_abstracts = 0

for entry in tqdm(ds_artificial, desc="Building PubMedQA RAG pool"):
    pubid = str(entry['pubid'])
    question = entry['question']
    long_answer = entry['long_answer']
    final_decision = entry['final_decision']

    retrieved_ids = query_to_retrieved_ids.get(pubid, [])

    # Filter to IDs that are actually present in the tokenized abstracts
    available_ids = [pid for pid in retrieved_ids if pid in abstract_list_tokenized]

    if not available_ids:
        skipped_no_abstracts += 1

    candidates = [
        {"sentence": sentence, "relevance_score": 0.0}
        for pid in available_ids
        for sentence in abstract_list_tokenized[pid]
    ]

    pubmedqa_rag_pool.append({
        "id": pubid,
        "query": question,
        "answer": long_answer,
        "final_decision": final_decision,
        "pubmed_ids": available_ids,
        "candidates": candidates
    })

print(f"Total pool entries      : {len(pubmedqa_rag_pool)}")
print(f"Entries with no abstracts: {skipped_no_abstracts}")
print(f"Entries with candidates : {len(pubmedqa_rag_pool) - skipped_no_abstracts}")

with open("pubmedqa_rag_pool.json", "w", encoding="utf-8") as f:
    json.dump(pubmedqa_rag_pool, f, ensure_ascii=False, indent=2)

print("RAG pool saved to pubmedqa_rag_pool.json")

Building PubMedQA RAG pool: 100%|██████████| 20/20 [00:00<00:00, 2977.32it/s]

Total pool entries      : 20
Entries with no abstracts: 12
Entries with candidates : 8
RAG pool saved to pubmedqa_rag_pool.json
